In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

#PATHS
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

In [3]:
# IDENTIFYING SCHEMA AND MISSING DATA
txn=pd.read_csv(RAW / "transactions.csv")
print(txn.shape)   #rows and columns
print(txn.dtypes)  #column data types
print(txn.head())  #first 5 rows
print(txn.isnull().sum())  #null count per column

(53704, 7)
Date                    object
SKU_ID                  object
Category                object
Listed_Price_INR       float64
Discount_Pct           float64
Effective_Price_INR    float64
Quantity_Sold            int64
dtype: object
         Date   SKU_ID     Category  Listed_Price_INR  Discount_Pct  \
0  2022-01-01  SKU0001  Electronics           9513.37           0.0   
1  2022-01-01  SKU0002  Electronics          20969.08           0.0   
2  2022-01-01  SKU0003  Electronics          36809.42           0.0   
3  2022-01-01  SKU0004  Electronics          11274.10           0.0   
4  2022-01-01  SKU0005  Electronics           6890.42           0.0   

   Effective_Price_INR  Quantity_Sold  
0              9513.37             26  
1             20969.08             18  
2             36809.42             25  
3             11274.10             17  
4              6890.42             21  
Date                   0
SKU_ID                 0
Category               0
Listed_Price_INR

In [4]:
# IDENTIFYING DUPLICATE ROWS
print("Duplicate Rows: ", txn.duplicated().sum())
print("Duplicate SKU IDs: ", txn.duplicated(subset=["SKU_ID","Date"]).sum())

Duplicate Rows:  0
Duplicate SKU IDs:  0


In [5]:
#STANDARDISE DATE
txn["Date"] = pd.to_datetime(txn["Date"], format='ISO8601').dt.strftime("%Y-%m-%d")
print(txn["Date"].head())
print(txn["Date"].dtype)

0    2022-01-01
1    2022-01-01
2    2022-01-01
3    2022-01-01
4    2022-01-01
Name: Date, dtype: object
object


In [6]:
# VALIDATING SELLING PRICE >= COST PRICE
master = pd.read_csv(RAW / "product_master.csv")

txn = txn.merge(master[["SKU_ID", "Base_Cost_INR"]], on="SKU_ID", how="left")

txn["price_violation"] = txn["Effective_Price_INR"] < txn["Base_Cost_INR"]
print("Price violations:", txn["price_violation"].sum())

violations = txn[txn["price_violation"]]
print(violations[["SKU_ID", "Effective_Price_INR", "Base_Cost_INR"]].head(20))

Price violations: 3417
      SKU_ID  Effective_Price_INR  Base_Cost_INR
931  SKU0001              7961.77           8000
932  SKU0002             16045.99          18000
933  SKU0003             29942.59          30000
934  SKU0004              8992.68           9000
935  SKU0005              5159.61           6000
936  SKU0006             36060.06          42000
937  SKU0007             11161.79          12000
938  SKU0008               542.45            550
941  SKU0011               471.55            520
942  SKU0012               351.99            360
943  SKU0013               253.67            280
945  SKU0015              1522.88           1600
949  SKU0019              1475.44           1500
950  SKU0020              4922.11           5000
952  SKU0022              3017.45           3200
954  SKU0024              1724.27           1800
955  SKU0025              1218.12           1400
957  SKU0027               490.28            500
960  SKU0030              2192.67           22

In [7]:
## CLEANED-TRANSACTIONS SAVED AS PARQUET
txn_clean = txn.drop_duplicates(subset=["SKU_ID", "Date"]) #0 duplicates, but defensive approach

# Ensuring the shape passed is clean. (Defensive approach)
print("Clean shape:", txn_clean.shape)
print("Price violations retained:", txn_clean["price_violation"].sum())

#Save to the processed folder
txn_clean.to_parquet(PROCESSED / "transactions_clean.parquet", index=False)
print("Saved: transactions_clean.parquet")

Clean shape: (53704, 9)
Price violations retained: 3417
Saved: transactions_clean.parquet
